# Coachable Robots: Edge-to-Cloud Training Pipeline

Benchmarking SO-ARM101 policy training with LeRobot on Chameleon Cloud MI100 GPUs.

## Architecture

```
┌─────────────────────┐     HuggingFace Hub      ┌──────────────────────────┐
│   Raspberry Pi 5    │  ──── dataset push ────>  │   Chameleon MI100 Node   │
│   (Edge Collector)  │                           │   (Training Server)      │
│                     │                           │                          │
│  xbox_soarm_teleop  │                           │  LeRobot + PyTorch ROCm  │
│  + 2x webcams       │                           │  ACT / Diffusion / Pi0   │
│  + SO-ARM101        │                           │                          │
│                     │  <── checkpoint pull ───  │  Trained policy ckpt     │
│  Docker container   │                           │                          │
│  (lerobot + teleop) │                           │  ROCm 6.3 + gfx908      │
└─────────────────────┘                           └──────────────────────────┘
```

## Notebook Goals

1. **Idempotent provisioning** — check for existing leases/servers before creating
2. **MI100 training environment** — ROCm + LeRobot on Chameleon bare metal
3. **Pi edge integration** — Docker container that collects, pushes, and fetches
4. **Benchmarking** — inference latency across hardware tiers

---
## Part 1: Chameleon Cloud Setup

In [1]:
import chi
from chi import lease, server, hardware
from chi.lease import Lease
from datetime import timedelta

# === CONFIGURE: Set your site, project, and resource names ===
chi.use_site("CHI@TACC")          # CHI@TACC or CHI@UC
chi.set("project_name", "CHI-XXXXXX")  # Your Chameleon allocation

# === Configuration ===
LEASE_NAME = "coachable-robots-mi100"
SERVER_NAME = "coachable-robots-training"
KEY_NAME = "YOUR_KEY_NAME"        # Key pair registered with Nova
NODE_TYPE = "gpu_mi100"
IMAGE_NAME = "CC-Ubuntu22.04"
LEASE_HOURS = 6

Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org


### 1a. Inspect Resources and Get/Create Lease

Checks for existing leases and servers. If a matching lease is found,
reuses it. If not, offers to create one. Never spawns duplicates.

In [2]:
my_lease = None

# ── Check for existing leases ──
print("=" * 60)
print("EXISTING LEASES")
print("=" * 60)

all_leases = lease.list_leases()
active_leases = [l for l in all_leases if l.status in ("ACTIVE", "PENDING")]

if active_leases:
    for i, l in enumerate(active_leases):
        marker = " <<<" if l.name == LEASE_NAME else ""
        print(f"  {i+1}. [{l.status}] {l.name}  (id: {l.id}){marker}")
        print(f"              ends: {l.end_date}")
        if l.node_reservations:
            print(f"              nodes: {len(l.node_reservations)} reservation(s)")
        if l.fip_reservations:
            print(f"              fips:  {len(l.fip_reservations)} reservation(s)")
        print()
        # Auto-select our named lease if it exists
        if l.name == LEASE_NAME:
            my_lease = l
else:
    print("  No active or pending leases.")

# ── Check for existing servers ──
print("=" * 60)
print("EXISTING SERVERS")
print("=" * 60)

try:
    existing_servers = server.list_servers()
    if existing_servers:
        for s in existing_servers:
            s_name = s.name if hasattr(s, 'name') else s.get('name', 'unknown')
            s_status = s.status if hasattr(s, 'status') else s.get('status', 'unknown')
            s_id = s.id if hasattr(s, 'id') else s.get('id', 'unknown')
            marker = " <<<" if s_name == SERVER_NAME else ""
            print(f"  [{s_status}] {s_name}  (id: {s_id}){marker}")
    else:
        print("  No servers running.")
except Exception as e:
    print(f"  Could not list servers: {e}")

# ── Check MI100 availability ──
print()
print("=" * 60)
print("MI100 AVAILABILITY")
print("=" * 60)
available_nodes = hardware.get_nodes(node_type=NODE_TYPE, filter_reserved=True)
print(f"  {len(available_nodes)} '{NODE_TYPE}' node(s) available for reservation.")

# ── Decision ──
print()
print("=" * 60)
if my_lease:
    print(f"REUSING lease '{my_lease.name}' [{my_lease.status}]")
    print(f"  ID:  {my_lease.id}")
    print(f"  End: {my_lease.end_date}")
else:
    print(f"No active lease named '{LEASE_NAME}'.")
    if not available_nodes:
        print(f"  WARNING: No {NODE_TYPE} nodes available either.")
        print("  Check the host calendar or try a different node type.")
    else:
        resp = input(f"Create a {LEASE_HOURS}h lease for 1x {NODE_TYPE}? [y/N]: ")
        if resp.strip().lower() in ('y', 'yes'):
            my_lease = Lease(
                name=LEASE_NAME,
                duration=timedelta(hours=LEASE_HOURS),
            )
            my_lease.add_node_reservation(node_type=NODE_TYPE, amount=1)
            my_lease.add_fip_reservation(amount=1)
            my_lease.submit(
                wait_for_active=True,
                wait_timeout=600,
                show="widget",
                idempotent=True,
            )
            print(f"\nLease ACTIVE: {my_lease.id}")
        else:
            print("Skipped lease creation. Re-run this cell when ready.")

EXISTING LEASES


Unauthorized: The request you have made requires authentication. (HTTP 401) (Request-ID: req-ece53eea-8f3a-4d8e-8a01-3c87bceaa62b)

In [4]:
import chi                                                                                                     
for site in ["CHI@TACC", "CHI@UC", "CHI@Edge"]:           
  try:                                                                                                       
      chi.use_site(site)
      from chi import lease                                                                                  
      lease.list_leases()                               
      print(f"{site}: ✅ auth OK")
  except Exception as e:                                                                                     
      print(f"{site}: ❌ {str(e)[:60]}")
                                               

Now using CHI@TACC:
URL: https://chi.tacc.chameleoncloud.org
Location: Austin, Texas, USA
Support contact: help@chameleoncloud.org
CHI@TACC: ❌ The request you have made requires authentication. (HTTP 401
Now using CHI@UC:
URL: https://chi.uc.chameleoncloud.org
Location: Argonne National Laboratory, Lemont, Illinois, USA
Support contact: help@chameleoncloud.org
CHI@UC: ❌ The request you have made requires authentication. (HTTP 401
Now using CHI@Edge:
URL: https://chi.edge.chameleoncloud.org
Location: University of Chicago, Chicago, Illinois, USA
Support contact: help@chameleoncloud.org
CHI@Edge: ❌ The request you have made requires authentication. (HTTP 401


### 1b. Create or Reuse Server

In [ ]:
import socket
import time

if my_lease is None:
    raise RuntimeError("No lease available. Run the previous cell and create one first.")

# Check if server already exists
gpu_server = None
floating_ip = None

try:
    existing_id = server.get_server_id(SERVER_NAME)
    gpu_server_obj = server.get_server(existing_id)
    status = gpu_server_obj.status if hasattr(gpu_server_obj, 'status') else gpu_server_obj.get('status')
    
    if status == "ACTIVE":
        print(f"Server '{SERVER_NAME}' already exists and is ACTIVE.")
        gpu_server = gpu_server_obj
    elif status == "BUILD":
        print(f"Server '{SERVER_NAME}' is still building. Waiting...")
        server.wait_for_active(existing_id)
        gpu_server = server.get_server(existing_id)
    else:
        print(f"Server '{SERVER_NAME}' exists but status is {status}. Deleting and recreating...")
        server.delete_server(existing_id)
        time.sleep(10)
except Exception:
    print(f"No existing server '{SERVER_NAME}'. Will create.")

# Create server if needed
if gpu_server is None:
    reservation_id = my_lease.node_reservations[0]["id"]
    print(f"Creating server with reservation {reservation_id}...")
    gpu_server = server.create_server(
        SERVER_NAME,
        reservation_id=reservation_id,
        image_name=IMAGE_NAME,
        key_name=KEY_NAME,
    )
    server.wait_for_active(gpu_server.id)
    print("Server is ACTIVE!")

# Attach or find floating IP
try:
    # Check if server already has a floating IP
    ips = server.list_floating_ips(gpu_server.id) if hasattr(server, 'list_floating_ips') else []
    if ips:
        floating_ip = ips[0]
        print(f"Existing floating IP: {floating_ip}")
except Exception:
    pass

if not floating_ip:
    floating_ip = server.associate_floating_ip(gpu_server.id)
    print(f"Assigned floating IP: {floating_ip}")

print(f"\nSSH: ssh cc@{floating_ip}")

### 1c. Wait for SSH and Verify MI100

In [ ]:
print(f"Waiting for SSH on {floating_ip}...")
timeout = 300
start = time.perf_counter()

while True:
    try:
        with socket.create_connection((floating_ip, 22), timeout=10):
            print("SSH ready!")
            break
    except OSError:
        elapsed = time.perf_counter() - start
        if elapsed >= timeout:
            print(f"Timed out after {timeout}s.")
            break
        print(f"  {elapsed:.0f}s... retrying")
        time.sleep(15)

In [ ]:
from chi import ssh

# MI100 is AMD — use lspci and rocm-smi, NOT nvidia-smi
with ssh.Remote(floating_ip) as conn:
    print("=== OS ===")
    conn.run("lsb_release -ds")
    
    print("\n=== AMD GPU Detection ===")
    conn.run("lspci | grep -i 'display\|vga\|amd\|radeon\|arcturus'")
    
    print("\n=== Kernel ===")
    conn.run("uname -r")
    
    print("\n=== Memory ===")
    conn.run("free -h | head -2")

---
## Part 2: Configure Node with Ansible

Uses an Ansible playbook to install ROCm + LeRobot. Idempotent —
safe to re-run if interrupted or if the node already has partial setup.

The playbook lives in `ansible/playbooks/setup_training_node.yml` and installs:
1. System deps (build tools, headers, tmux)
2. ROCm 6.3 (skips if already installed)
3. Miniconda (skips if already installed)
4. PyTorch 2.7.1 + ROCm 6.3 in a `lerobot` conda env (Python 3.12)
5. LeRobot v0.5.0 + HuggingFace CLI (`hf`)

In [ ]:
import subprocess, os

# Path to the ansible directory in your Chameleon Jupyter workspace
ANSIBLE_DIR = "/work/projects/coachable-robots/ansible"
PRIVATE_KEY = "/work/.ssh/id_rsa"
VAULT_PASSWORD_FILE = os.path.join(ANSIBLE_DIR, ".vault_pass")
# Create with: echo 'yourpassword' > ansible/.vault_pass && chmod 600 ansible/.vault_pass

# Generate inventory file with the current floating IP
inventory_content = f"""[training]
mi100 ansible_host={floating_ip} ansible_user=cc ansible_ssh_private_key_file={PRIVATE_KEY}

[training:vars]
ansible_ssh_common_args=-o StrictHostKeyChecking=no
"""

inventory_path = os.path.join(ANSIBLE_DIR, "inventory.ini")
os.makedirs(ANSIBLE_DIR, exist_ok=True)
with open(inventory_path, "w") as f:
    f.write(inventory_content)

print(f"Inventory written to {inventory_path}")
print(f"  Target: cc@{floating_ip}")
print(f"  Key:    {PRIVATE_KEY}")
print()
print("Run the next cell to execute the playbook, or run manually:")
print(f"  cd {ANSIBLE_DIR}")
print(f"  ansible-playbook -i inventory.ini playbooks/setup_training_node.yml --vault-password-file .vault_pass")

In [ ]:
# Run the Ansible playbook (15-20 min on first run, fast on re-runs)
# Streams output in real time so you can watch progress.

playbook_path = os.path.join(ANSIBLE_DIR, "playbooks/setup_training_node.yml")

cmd = [
    "ansible-playbook",
    "-i", inventory_path,
    playbook_path,
    "--vault-password-file", VAULT_PASSWORD_FILE,
    "-v",  # verbose — remove for less output
]

print(f"Running: {' '.join(cmd)}")
print("=" * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=ANSIBLE_DIR,
)

for line in proc.stdout:
    print(line, end="")

rc = proc.wait()
print("=" * 60)
if rc == 0:
    print("Playbook completed successfully.")
else:
    print(f"Playbook failed with exit code {rc}.")
    print("Re-run this cell to retry — it will skip completed steps.")

---
## Part 3: Raspberry Pi Edge Container

The Pi runs a Docker container that handles:
1. **Data collection** — xbox_soarm_teleop + cameras → LeRobot dataset
2. **Dataset push** — upload to HuggingFace Hub
3. **Checkpoint fetch** — pull trained policy back for inference

### Why HuggingFace Hub instead of Chameleon Object Store?

LeRobot datasets are natively HF datasets. The `lerobot-record` command
already supports `--dataset.push_to_hub=true`. Using HF Hub means:
- Zero custom transfer code
- Training script pulls data with just a `repo_id`
- Checkpoints push/pull the same way
- Students can browse their datasets on huggingface.co

Chameleon Object Store is still useful for large artifacts, snapshots,
and Trovi packaging of the full experiment.

### Pi Container Image

The Pi uses a pre-built arm64 Docker image: `rianders/lerobot-soarm101:latest`

Built from [`docker/Dockerfile.pi`](docker/Dockerfile.pi) with:
- `python:3.12-slim-bookworm` (arm64)
- CPU-only PyTorch (Pi 5 has no GPU)
- LeRobot v0.5.0 with Feetech servo support
- Gradio for camera preview
- `uv` as package manager
- Scripts in `/app/scripts/`

To rebuild locally (cross-compile for arm64):
```bash
docker buildx build --platform linux/arm64 --load \
  -t rianders/lerobot-soarm101:latest -f docker/Dockerfile.pi .
docker push rianders/lerobot-soarm101:latest
```

In [ ]:
import subprocess, shlex

# === CONFIGURE ===
PI_IP = "192.168.4.191"
PI_SSH_PORT = 22222
PI_USER = "root"
PI_IMAGE = "rianders/lerobot-soarm101:latest"

def pi_run(cmd, capture=False):
    """Run a command on the Pi over SSH."""
    ssh_cmd = f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_IP} {shlex.quote(cmd)}"
    result = subprocess.run(ssh_cmd, shell=True, capture_output=capture, text=True)
    if not capture:
        return result.returncode
    return result.stdout.strip()

# ── Pull latest image on Pi ──
print(f"Pulling {PI_IMAGE} on Pi...")
pi_run(f"balena pull {PI_IMAGE}")

# ── Verify image and smoke test ──
print("\nVerifying image...")
out = pi_run(
    f"balena run --rm {PI_IMAGE} "
    f"python -c \"import torch, lerobot; print('torch:', torch.__version__, '| lerobot:', lerobot.__version__)\"",
    capture=True
)
print(out)

# ── Show connected devices ──
print("\nConnected devices:")
print("  Serial:", pi_run("ls /dev/ttyACM* 2>/dev/null || echo 'none'", capture=True))
print("  Cameras:", pi_run("ls /dev/video0 /dev/video1 2>/dev/null || echo 'none'", capture=True))

### 3a. Camera Preview

Before calibrating, verify the camera view and adjust framing so the workspace is fully visible.

The Pi container runs a Gradio preview app on port 7860. The cell below:
1. Starts the preview container on the Pi (or restarts if already running)
2. Opens an SSH tunnel to forward port 7860 locally
3. Embeds the live feed inline in this notebook

**Adjust `CAMERA_INDEX`** if you need a different camera (`0` = Logitech C920).  
Set `PI_IP` and `PI_SSH_PORT` to match your Pi's address.

In [ ]:
import subprocess, time, shlex

# === CONFIGURE ===
PI_IP = "192.168.4.191"
PI_SSH_PORT = 22222
PI_USER = "root"
PI_IMAGE = "rianders/lerobot-soarm101:latest"
CAMERA_INDEX = 0          # 0 = Logitech C920 on /dev/video0
PREVIEW_PORT = 7860       # Local port to forward (and container port)

# ── Stop any existing preview container on the Pi ──
print("Stopping any existing camera preview container...")
subprocess.run(
    shlex.split(
        f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_IP} "
        f'"balena ps -q --filter ancestor={PI_IMAGE} | xargs -r balena stop"'
    ), capture_output=True
)

# ── Start the preview container ──
print("Starting camera preview container on Pi...")
result = subprocess.run(
    shlex.split(
        f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_IP} "
        f'"balena run -d --rm --privileged -v /dev:/dev '
        f'-p {PREVIEW_PORT}:{PREVIEW_PORT} {PI_IMAGE} '
        f'python scripts/camera_preview.py --camera {CAMERA_INDEX}"'
    ),
    capture_output=True, text=True
)
container_id = result.stdout.strip()
print(f"Container: {container_id[:12] if container_id else 'unknown'}")

# ── Open SSH tunnel ──
print(f"Opening SSH tunnel: localhost:{PREVIEW_PORT} → Pi:{PREVIEW_PORT}")
tunnel = subprocess.Popen(
    shlex.split(
        f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no "
        f"-L {PREVIEW_PORT}:localhost:{PREVIEW_PORT} -N {PI_USER}@{PI_IP}"
    )
)

# Wait for Gradio to start up
print("Waiting for Gradio to start...")
time.sleep(8)
print(f"Preview ready — see embedded view below.")

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=f"http://localhost:{PREVIEW_PORT}", width="100%", height=600))

# To stop the preview when done:
#   tunnel.terminate()
#   subprocess.run(shlex.split(
#       f"ssh -p {PI_SSH_PORT} -o StrictHostKeyChecking=no {PI_USER}@{PI_IP} "
#       f'"balena ps -q --filter ancestor={PI_IMAGE} | xargs -r balena stop"'
#   ))

---
## Part 4: Training on MI100

Once the Pi has pushed a dataset, trigger training on the Chameleon node.

**MI100 (gfx908) constraints:**
- No Flash Attention 2 (gfx90a+ only)
- No hipBLASLt
- 32 GB HBM2 — plenty for ACT, tight for Pi0 3B
- ACT is recommended for MI100; Pi0 needs gradient checkpointing + bf16

In [ ]:
# Trigger training on the remote MI100 node
# Replace HF_USER and dataset name with your values

HF_USER = "YOUR_HF_USERNAME"  # CONFIGURE: your HuggingFace username
DATASET = "soarm101_demos"  # Dataset pushed from Pi
POLICY = "act"  # 'act' for MI100, 'pi0' for H100

train_cmd = f"""
source ~/miniconda3/bin/activate lerobot
cd ~/lerobot

python lerobot/scripts/train.py \\
    --dataset.repo_id={HF_USER}/{DATASET} \\
    --policy.path=lerobot/{POLICY} \\
    --output_dir=outputs/train/{POLICY}_{DATASET} \\
    --job_name={POLICY}_{DATASET} \\
    --policy.device=cuda \\
    --wandb.enable=false
"""

print("Training command (run on MI100 node):")
print(train_cmd)
print(f"To execute: ssh cc@{floating_ip} and paste the above")
print("Or run the next cell to launch it remotely.")

In [ ]:
# Optional: launch training remotely (runs in foreground — long!)
# Consider using tmux or nohup via SSH for production runs.
#
# with ssh.Remote(floating_ip) as conn:
#     conn.run(train_cmd)

---
## Part 5: Benchmark Inference Latency

For coachable-robots-bench, record inference timing on each tier.

In [ ]:
benchmark_script = """
import torch, time, json, platform

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device_name = torch.cuda.get_device_name(0) if device == 'cuda' else platform.processor()

results = {'device': device_name, 'pytorch': torch.__version__, 'tests': []}

for label, shape in [('policy_input_480p', (1, 3, 480, 640)), ('policy_input_224', (1, 3, 224, 224))]:
    x = torch.randn(*shape, device=device)
    # Warmup
    for _ in range(20):
        _ = torch.nn.functional.interpolate(x, size=(224, 224), mode='bilinear')
    if device == 'cuda': torch.cuda.synchronize()
    
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        _ = torch.nn.functional.interpolate(x, size=(224, 224), mode='bilinear')
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    
    results['tests'].append({
        'name': label, 'shape': list(shape),
        'avg_ms': round(sum(times)/len(times), 3),
        'p50_ms': round(sorted(times)[len(times)//2], 3),
        'p99_ms': round(sorted(times)[int(len(times)*0.99)], 3),
    })

print(json.dumps(results, indent=2))
"""

with ssh.Remote(floating_ip) as conn:
    conn.run(f"""
        source ~/miniconda3/bin/activate lerobot
        python3 -c '{benchmark_script}'
    """)

---
## Part 6: Cleanup

**Run this when done** to release bare-metal resources and stop burning SUs.

In [ ]:
# Safety: show what we're about to delete
print("Will delete:")
print(f"  Server: {SERVER_NAME}")
print(f"  Lease:  {LEASE_NAME} (id: {my_lease.id})")
print()
confirm = input("Type 'yes' to confirm cleanup: ")

if confirm.strip().lower() == 'yes':
    try:
        sid = server.get_server_id(SERVER_NAME)
        server.delete_server(sid)
        print(f"Server '{SERVER_NAME}' deleted.")
    except Exception as e:
        print(f"Server cleanup: {e}")

    try:
        lease.delete_lease(my_lease.id)
        print(f"Lease '{LEASE_NAME}' deleted. Hardware released.")
    except Exception as e:
        print(f"Lease cleanup: {e}")
    
    print("\nCleanup complete.")
else:
    print("Cleanup cancelled.")